# Human Annotation

### Purpose
This notebook is designed to validate the reliability of LLM evaluation scores. 

We will annotate a sample of QA pairs that received perfect scores (2/2/2) 

from the LLM evaluator to verify if these scores are accurate.

### Overview
- **Total QA pairs**: 1,483
- **Perfect-score QA pairs**: 1,087 (73.3%)
- **Sampled for annotation**: 108 QA pairs (10% of perfect scores)

All selected QA pairs have LLM scores of:
- Question Score: 2/2
- Answer Score: 2/2  
- Choice Score: 2/2

Your task is to evaluate these QA pairs using the same criteria and compare your scores with the LLM scores.

## Cell 1: Data Loading and Sampling

**Run this cell first!**

This cell:
1. Loads all QA pairs from the `data_qa_with_llmeval` directory
2. Filters for QA pairs with perfect LLM scores (all scores = 2)
3. Randomly samples 10% of perfect-score pairs for annotation
4. Creates annotation files (JSON and CSV) in the `./annotation_data/` directory

**Output files:**
- `sampled_perfect_score_qa_for_annotation.json` - Full data with all context
- `sampled_perfect_score_qa_annotation.csv` - Flattened CSV for easy annotation

After running this cell, you'll see statistics about the sampled data.

In [1]:
import json
import os
import pandas as pd
from pathlib import Path
from typing import List, Dict, Any
import glob

# Configuration
DATA_DIR = '../data_qa_with_llmeval'
OUTPUT_DIR = './annotation_data'
PROGRESS_FILE = './annotation_progress.json'

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

def load_all_qa_data(data_dir: str) -> List[Dict[str, Any]]:
    """
    Load all QA pairs from all files in data_qa_with_llmeval directory.
    Returns a list of QA pairs with full context.
    """
    all_qa_pairs = []
    
    # Find all JSON files in the directory
    pattern = os.path.join(data_dir, '*_with_llmeval.json')
    files = glob.glob(pattern)
    
    print(f"Found {len(files)} files to process")
    
    for file_path in sorted(files):
        filename = os.path.basename(file_path)
        print(f"Processing {filename}...")
        
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            events = data.get('events', [])
            
            for event_idx, event in enumerate(events):
                event_date = event.get('event_date', 'Unknown')
                event_sum = event.get('event_sum', '')
                event_text = event.get('event_text', [])
                structured_data = event.get('structured_data', {})
                
                qa_pairs = event.get('qa_pairs', [])
                
                for qa_idx, qa_pair in enumerate(qa_pairs):
                    # Extract LLM evaluation if available
                    llm_eval = qa_pair.get('llm_evaluation', {})
                    
                    # Create a comprehensive annotation entry
                    annotation_entry = {
                        # Identification
                        'file_name': filename,
                        'event_idx': event_idx,
                        'qa_idx': qa_idx,
                        'unique_id': f"{filename}_{event_idx}_{qa_idx}",
                        
                        # Event context
                        'event_date': event_date,
                        'event_sum': event_sum,
                        'event_text': event_text,  # List of articles
                        'referenced_article': qa_pair.get('referenced_article', ''),
                        
                        # QA pair
                        'question_type': qa_pair.get('question_type', ''),
                        'question': qa_pair.get('question', ''),
                        'answer': qa_pair.get('answer', ''),
                        'choices': qa_pair.get('choices', {}),
                        'candidate_choices': qa_pair.get('candidate_choices', []),
                        
                        # LLM evaluation scores
                        'llm_question_score': llm_eval.get('question_score', None),
                        'llm_answer_score': llm_eval.get('answer_score', None),
                        'llm_choice_score': llm_eval.get('choice_score', None),
                        'llm_total_score': (
                            (llm_eval.get('question_score', 0) or 0) +
                            (llm_eval.get('answer_score', 0) or 0) +
                            (llm_eval.get('choice_score', 0) or 0)
                        ),
                        
                        # LLM evaluation details
                        'llm_question_details': llm_eval.get('question_quality', {}),
                        'llm_answer_details': llm_eval.get('answer_quality', {}),
                        'llm_choice_details': llm_eval.get('choice_quality', {}),
                        
                        # Structured data
                        'people': structured_data.get('People', []),
                        'locations': structured_data.get('Location', []),
                        'results': structured_data.get('Result', []),
                        
                        # Human annotation fields (to be filled)
                        'human_question_score': None,
                        'human_answer_score': None,
                        'human_choice_score': None,
                        'human_notes': '',
                        'human_annotated': False,
                        'human_annotator': '',
                        'annotation_date': None
                    }
                    
                    all_qa_pairs.append(annotation_entry)
        
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    print(f"\nTotal QA pairs loaded: {len(all_qa_pairs)}")
    return all_qa_pairs

# Load all data
all_qa_data = load_all_qa_data(DATA_DIR)

# Filter for perfect-score QA pairs (question_score=2, answer_score=2, choice_score=2)
print("\n" + "="*80)
print("FILTERING FOR PERFECT SCORE QA PAIRS")
print("="*80)
perfect_score_qa = []
for entry in all_qa_data:
    q_score = entry.get('llm_question_score')
    a_score = entry.get('llm_answer_score')
    c_score = entry.get('llm_choice_score')
    
    if q_score == 2 and a_score == 2 and c_score == 2:
        perfect_score_qa.append(entry)

print(f"Total QA pairs: {len(all_qa_data)}")
print(f"Perfect score QA pairs (all scores = 2): {len(perfect_score_qa)}")
print(f"Percentage: {len(perfect_score_qa)/len(all_qa_data)*100:.1f}%")

# Sample 10% of perfect-score QA pairs for human annotation
import random
random.seed(42)  # Set seed for reproducibility

sample_size = max(1, int(len(perfect_score_qa) * 0.10))
sampled_qa = random.sample(perfect_score_qa, sample_size)

print(f"\nSampled {sample_size} QA pairs ({sample_size/len(perfect_score_qa)*100:.1f}% of perfect scores)")
print(f"This represents {sample_size/len(all_qa_data)*100:.2f}% of all QA pairs")

# Use sampled data for annotation
all_qa_data = sampled_qa

# Save as JSON for easy access
output_json = os.path.join(OUTPUT_DIR, 'sampled_perfect_score_qa_for_annotation.json')
with open(output_json, 'w', encoding='utf-8') as f:
    json.dump(all_qa_data, f, indent=2, ensure_ascii=False)

print(f"\nSaved sampled QA pairs to: {output_json}")

# Create a flattened CSV for easier annotation
def create_annotation_csv(qa_data: List[Dict[str, Any]], output_path: str):
    """
    Create a CSV file with flattened structure for easier annotation.
    """
    rows = []
    
    for entry in qa_data:
        # Flatten choices into separate columns
        choices = entry.get('choices', {})
        choice_a = choices.get('a', '')
        choice_b = choices.get('b', '')
        choice_c = choices.get('c', '')
        choice_d = choices.get('d', '')
        
        # Get answer text
        answer_label = entry.get('answer', '')
        answer_text = choices.get(answer_label, '')
        
        # Flatten event text (join articles)
        event_text_str = '\n\n---\n\n'.join(entry.get('event_text', []))
        
        row = {
            'unique_id': entry['unique_id'],
            'file_name': entry['file_name'],
            'event_idx': entry['event_idx'],
            'qa_idx': entry['qa_idx'],
            'event_date': entry['event_date'],
            'event_sum': entry['event_sum'],
            'referenced_article': entry['referenced_article'],
            'question_type': entry['question_type'],
            'question': entry['question'],
            'answer_label': answer_label,
            'answer_text': answer_text,
            'choice_a': choice_a,
            'choice_b': choice_b,
            'choice_c': choice_c,
            'choice_d': choice_d,
            'llm_question_score': entry['llm_question_score'],
            'llm_answer_score': entry['llm_answer_score'],
            'llm_choice_score': entry['llm_choice_score'],
            'llm_total_score': entry['llm_total_score'],
            'event_text': event_text_str[:5000],  # Truncate for CSV readability
            'people': ', '.join(entry.get('people', [])),
            'locations': ', '.join(entry.get('locations', [])),
            'results': ', '.join(entry.get('results', [])),
            # Human annotation columns
            'human_question_score': entry['human_question_score'],
            'human_answer_score': entry['human_answer_score'],
            'human_choice_score': entry['human_choice_score'],
            'human_notes': entry['human_notes'],
            'human_annotated': entry['human_annotated'],
            'human_annotator': entry['human_annotator'],
            'annotation_date': entry['annotation_date']
        }
        rows.append(row)
    
    df = pd.DataFrame(rows)
    df.to_csv(output_path, index=False, encoding='utf-8')
    return df

# Create CSV
output_csv = os.path.join(OUTPUT_DIR, 'sampled_perfect_score_qa_annotation.csv')
df = create_annotation_csv(all_qa_data, output_csv)
print(f"Saved CSV to: {output_csv}")
print(f"\nDataFrame shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")

# Display summary statistics
print("\n" + "="*80)
print("SUMMARY STATISTICS FOR SAMPLED DATA")
print("="*80)
print(f"Total QA pairs selected for annotation: {len(all_qa_data)}")
print(f"Files represented: {len(set(e['file_name'] for e in all_qa_data))}")
print(f"Events represented: {len(set((e['file_name'], e['event_idx']) for e in all_qa_data))}")

# Verify all have perfect scores
all_perfect = all(
    e.get('llm_question_score') == 2 and 
    e.get('llm_answer_score') == 2 and 
    e.get('llm_choice_score') == 2 
    for e in all_qa_data
)
print(f"\nAll selected QA pairs have perfect LLM scores: {all_perfect}")

# Distribution by file
print(f"\nDistribution by file:")
file_counts = {}
for e in all_qa_data:
    file_counts[e['file_name']] = file_counts.get(e['file_name'], 0) + 1
for filename, count in sorted(file_counts.items()):
    print(f"  {filename}: {count}")

print("\n" + "="*80)
print("SETUP COMPLETE!")
print("="*80)
print(f"\nPurpose: Validate LLM evaluation reliability")
print(f"Selected: {len(all_qa_data)} QA pairs (10% of perfect-score pairs)")
print(f"\nNext steps:")
print(f"1. Review the CSV file: {output_csv}")
print(f"2. Annotate using the 'human_*' columns")
print(f"3. Compare human scores with LLM scores (all should be 2/2/2)")
print(f"4. Use the progress tracking functions below to save your work")

Found 6 files to process
Processing temporal_qa_1_with_llmeval.json...
Processing temporal_qa_218_with_llmeval.json...
Processing temporal_qa_227_with_llmeval.json...
Processing temporal_qa_25_with_llmeval.json...
Processing temporal_qa_2_with_llmeval.json...
Processing temporal_qa_3_with_llmeval.json...

Total QA pairs loaded: 1483

FILTERING FOR PERFECT SCORE QA PAIRS
Total QA pairs: 1483
Perfect score QA pairs (all scores = 2): 1087
Percentage: 73.3%

Sampled 108 QA pairs (9.9% of perfect scores)
This represents 7.28% of all QA pairs

Saved sampled QA pairs to: ./annotation_data/sampled_perfect_score_qa_for_annotation.json
Saved CSV to: ./annotation_data/sampled_perfect_score_qa_annotation.csv

DataFrame shape: (108, 30)

Columns: ['unique_id', 'file_name', 'event_idx', 'qa_idx', 'event_date', 'event_sum', 'referenced_article', 'question_type', 'question', 'answer_label', 'answer_text', 'choice_a', 'choice_b', 'choice_c', 'choice_d', 'llm_question_score', 'llm_answer_score', 'llm_ch

## Cell 2: Helper Functions for Annotation

**Run this cell to load helper functions.**

This cell defines utility functions for:
- **`update_annotation()`** - Update a single QA pair with your scores
- **`save_annotations()`** - Save your annotations to files
- **`get_unannotated_samples()`** - Get random samples of unannotated pairs
- **`display_qa_for_annotation()`** - Pretty-print a QA pair for review
- **`load_progress()` / `save_progress()`** - Track annotation progress

You'll use these functions in the next cell to annotate QA pairs.

In [19]:
# Helper functions for annotation progress tracking

def load_progress(progress_file: str = PROGRESS_FILE) -> Dict[str, Any]:
    """Load annotation progress from file."""
    if os.path.exists(progress_file):
        with open(progress_file, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {}

def save_progress(progress: Dict[str, Any], progress_file: str = PROGRESS_FILE):
    """Save annotation progress to file."""
    with open(progress_file, 'w', encoding='utf-8') as f:
        json.dump(progress, f, indent=2, ensure_ascii=False)

def update_annotation(unique_id: str, 
                     question_score: int = None,
                     answer_score: int = None,
                     choice_score: int = None,
                     notes: str = '',
                     annotator: str = '',
                     qa_data: List[Dict[str, Any]] = None,
                     progress_file: str = PROGRESS_FILE):
    """
    Update a single QA pair annotation.
    
    Args:
        unique_id: Unique identifier for the QA pair
        question_score: Human annotation score for question (0-2)
        answer_score: Human annotation score for answer (0-2)
        choice_score: Human annotation score for choices (0-2)
        notes: Optional notes
        annotator: Name of annotator
        qa_data: The full QA data list (will update in place)
        progress_file: Path to progress file
    """
    from datetime import datetime
    
    # Find the entry
    entry = None
    if qa_data:
        for e in qa_data:
            if e['unique_id'] == unique_id:
                entry = e
                break
    
    if not entry:
        print(f"Warning: Entry {unique_id} not found")
        return
    
    # Update fields
    if question_score is not None:
        entry['human_question_score'] = question_score
    if answer_score is not None:
        entry['human_answer_score'] = answer_score
    if choice_score is not None:
        entry['human_choice_score'] = choice_score
    if notes:
        entry['human_notes'] = notes
    if annotator:
        entry['human_annotator'] = annotator
    
    # Mark as annotated if all scores provided
    if (entry['human_question_score'] is not None and 
        entry['human_answer_score'] is not None and 
        entry['human_choice_score'] is not None):
        entry['human_annotated'] = True
        entry['annotation_date'] = datetime.now().isoformat()
    
    # Update progress file
    progress = load_progress(progress_file)
    progress[unique_id] = {
        'question_score': entry['human_question_score'],
        'answer_score': entry['human_answer_score'],
        'choice_score': entry['human_choice_score'],
        'notes': entry['human_notes'],
        'annotator': entry['human_annotator'],
        'annotated': entry['human_annotated'],
        'annotation_date': entry['annotation_date']
    }
    save_progress(progress, progress_file)
    
    print(f"Updated annotation for {unique_id}")

def save_annotations(qa_data: List[Dict[str, Any]], 
                    output_path: str = None,
                    format: str = 'both'):
    """
    Save annotated data back to files.
    
    Args:
        qa_data: The full QA data list with annotations
        output_path: Output directory (default: OUTPUT_DIR)
        format: 'json', 'csv', or 'both'
    """
    from datetime import datetime
    
    if output_path is None:
        output_path = OUTPUT_DIR
    
    os.makedirs(output_path, exist_ok=True)
    
    if format in ['json', 'both']:
        # Save as JSON
        json_path = os.path.join(output_path, 'annotated_qa_pairs.json')
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(qa_data, f, indent=2, ensure_ascii=False)
        print(f"Saved annotated JSON to: {json_path}")
    
    if format in ['csv', 'both']:
        # Save as CSV
        csv_path = os.path.join(output_path, 'annotated_qa_pairs.csv')
        df = create_annotation_csv(qa_data, csv_path)
        print(f"Saved annotated CSV to: {csv_path}")
    
    # Save progress summary
    annotated_count = sum(1 for e in qa_data if e.get('human_annotated', False))
    total_count = len(qa_data)
    
    summary = {
        'total_qa_pairs': total_count,
        'annotated_count': annotated_count,
        'remaining_count': total_count - annotated_count,
        'completion_percentage': (annotated_count / total_count * 100) if total_count > 0 else 0,
        'last_updated': datetime.now().isoformat()
    }
    
    summary_path = os.path.join(output_path, 'annotation_summary.json')
    with open(summary_path, 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    
    print(f"\nAnnotation Summary:")
    print(f"  Annotated: {annotated_count}/{total_count} ({summary['completion_percentage']:.1f}%)")
    print(f"  Remaining: {total_count - annotated_count}")

def get_unannotated_samples(qa_data: List[Dict[str, Any]], 
                           n: int = 10,
                           filter_by_llm_score: bool = False,
                           min_llm_score: int = None,
                           max_llm_score: int = None):
    """
    Get a sample of unannotated QA pairs for annotation.
    
    Args:
        qa_data: The full QA data list
        n: Number of samples to return
        filter_by_llm_score: Whether to filter by LLM score
        min_llm_score: Minimum LLM total score
        max_llm_score: Maximum LLM total score
    """
    unannotated = [e for e in qa_data if not e.get('human_annotated', False)]
    
    if filter_by_llm_score:
        if min_llm_score is not None:
            unannotated = [e for e in unannotated if e.get('llm_total_score', 0) >= min_llm_score]
        if max_llm_score is not None:
            unannotated = [e for e in unannotated if e.get('llm_total_score', 0) <= max_llm_score]
    
    import random
    sample = random.sample(unannotated, min(n, len(unannotated)))
    
    return sample

def display_qa_for_annotation(qa_entry: Dict[str, Any]):
    """
    Pretty print a QA pair for annotation.
    """
    print("="*80)
    print(f"ID: {qa_entry['unique_id']}")
    print(f"File: {qa_entry['file_name']} | Event: {qa_entry['event_idx']} | QA: {qa_entry['qa_idx']}")
    print("="*80)
    print(f"\nEvent Date: {qa_entry['event_date']}")
    print(f"Event Summary: {qa_entry['event_sum']}")
    print(f"\nQuestion Type: {qa_entry['question_type']}")
    print(f"\nQuestion: {qa_entry['question']}")
    print(f"\nChoices:")
    choices = qa_entry['choices']
    for label in ['a', 'b', 'c', 'd']:
        marker = "✓" if label == qa_entry['answer'] else " "
        print(f"  {marker} {label.upper()}: {choices.get(label, '')}")
    print(f"\nCorrect Answer: {qa_entry['answer'].upper()}")
    print(f"\nReferenced Article: {qa_entry['referenced_article']}")
    print(f"\nLLM Scores: Q={qa_entry['llm_question_score']}, A={qa_entry['llm_answer_score']}, C={qa_entry['llm_choice_score']} (Total: {qa_entry['llm_total_score']})")
    print(f"\nLLM Reasoning for Answer Concealment: {qa_entry['llm_question_details']['answer_concealment']}")
    print(f"\nLLM Reasoning for Temporal Validity: {qa_entry['llm_question_details']['temporal_validity']}")
    print(f"\nLLM Reasoning for Contextual Groundedness: {qa_entry['llm_answer_details']['contextual_groundedness']}")
    print(f"\nLLM Reasoning for Specificity Alignment: {qa_entry['llm_answer_details']['specificity_alignment']}")
    print(f"\nLLM Reasoning for Contextual Plausibility: {qa_entry['llm_choice_details']['contextual_plausibility']}")
    print(f"\nLLM Reasoning for Mutual Exclusivity: {qa_entry['llm_choice_details']['mutual_exclusivity']}")
    if qa_entry.get('human_annotated'):
        print(f"\nHuman Scores: Q={qa_entry['human_question_score']}, A={qa_entry['human_answer_score']}, C={qa_entry['human_choice_score']}")
        if qa_entry.get('human_notes'):
            print(f"Notes: {qa_entry['human_notes']}")
    print("="*80)

# Example usage:
print("Helper functions loaded!")
print("\nExample usage:")
print("  # Get 5 unannotated samples")
print("  samples = get_unannotated_samples(all_qa_data, n=5)")
print("  # Display one for annotation")
print("  display_qa_for_annotation(samples[0])")
print("  # Update annotation")
print("  update_annotation(samples[0]['unique_id'], question_score=2, answer_score=1, choice_score=2, annotator='YourName')")
print("  # Save all annotations")
print("  save_annotations(all_qa_data)")

Helper functions loaded!

Example usage:
  # Get 5 unannotated samples
  samples = get_unannotated_samples(all_qa_data, n=5)
  # Display one for annotation
  display_qa_for_annotation(samples[0])
  # Update annotation
  update_annotation(samples[0]['unique_id'], question_score=2, answer_score=1, choice_score=2, annotator='YourName')
  # Save all annotations
  save_annotations(all_qa_data)


## Cell 3: Quick Start - Annotation Workflow

**Use this cell to start annotating!**

This cell demonstrates how to:
1. Get a sample of QA pairs to annotate
2. Display a QA pair for review
3. Update annotations with your scores
4. Track your progress

### Annotation Criteria

**Question Quality (0-2, 1 point for each criterion)**
* Answer Concealment : Question should not include the answer target.
* Temporal Validity: Question must target time-varying attributes or causal relationships and be explicitly tied to a temporal reference (e.g., currently, after ~, etc.).

**Answer Quality (0-2, 1 point for each criterion)**
* Contextual Groundedness: Answer should be grounded in the Event Context.
* Specificity Alignment: The specificity of the answer should be aligned to the specificity of the reported information.

**Choice Quality (0-2, 1 point for each criterion)**
* Contextual Plausibility: Choices must be contextually relevant and match the specificity level of the answer.
* Mutual Exclusivity: Choices should be mutually exclusive, ensuring no redundancy or semantic overlap.

---

*Remember*: All QA pairs here have LLM scores of 2/2/2. Your task is to verify if these scores are correct. If your scores differ, note the discrepancy in the notes field (`entry["human_notes"]`).

In [20]:
# NOTE: All QA pairs in this dataset have perfect LLM scores (2/2/2)
# Purpose: Validate whether LLM evaluation is reliable

# 2. Display the first QA pair for annotation
print("Displaying first QA pair:")
display_qa_for_annotation(all_qa_data[75])

# update_annotation(
#     all_qa_data[74]['unique_id'],
#     question_score=2,  # 0-2: 0=both criteria failed, 1=one criterion met, 2=both criteria met
#     answer_score=2,     # 0-2: 0=both criteria failed, 1=one criterion met, 2=both criteria met
#     choice_score=2,     # 0-2: 0=both criteria failed, 1=one criterion met, 2=both criteria met
#     notes='there is a word (his) that implies the answer',
#     annotator='Woojun',
#     qa_data=all_qa_data
# )

print("\n" + "="*80)
print("ANNOTATION CRITERIA REMINDER:")
print("="*80)
print("Question Quality (0-2):")
print("  - Answer Concealment: Question should not include the answer target")
print("  - Temporal Validity: Question must target time-varying attributes or causal relationships with temporal reference")
print("\nAnswer Quality (0-2):")
print("  - Contextual Groundedness: Answer should be grounded in Event Context")
print("  - Specificity Alignment: Answer specificity should match reported information")
print("\nChoice Quality (0-2):")
print("  - Contextual Plausibility: Choices must be contextually relevant")
print("  - Mutual Exclusivity: Choices should be mutually exclusive")
print("\n" + "="*80)
print("VALIDATION PURPOSE:")
print("="*80)
print("All QA pairs here have LLM scores of 2/2/2 (perfect scores).")
print("Your task is to verify if the LLM evaluation is correct.")
print("If your scores match (2/2/2), the LLM evaluation is reliable.")
print("If your scores differ, note the discrepancy in the notes field.")
print("="*80)

# 3. Check current annotation progress
annotated = sum(1 for e in all_qa_data if e.get('human_annotated', False))
total = len(all_qa_data)
print(f"\nCurrent Progress: {annotated}/{total} annotated ({annotated/total*100:.1f}%)")

# 4. Calculate agreement statistics (if any annotations exist)
if annotated > 0:
    agreements = 0
    disagreements = 0
    for e in all_qa_data:
        if e.get('human_annotated', False):
            llm_q = e.get('llm_question_score')
            llm_a = e.get('llm_answer_score')
            llm_c = e.get('llm_choice_score')
            human_q = e.get('human_question_score')
            human_a = e.get('human_answer_score')
            human_c = e.get('human_choice_score')
            
            if (llm_q == human_q and llm_a == human_a and llm_c == human_c):
                agreements += 1
            else:
                disagreements += 1
    
    print(f"\nAgreement Analysis:")
    print(f"  Full agreement (human = LLM): {agreements}/{annotated} ({agreements/annotated*100:.1f}%)")
    print(f"  Disagreements: {disagreements}/{annotated} ({disagreements/annotated*100:.1f}%)")

# 5. Save annotations (run this after annotating)
# save_annotations(all_qa_data)

Displaying first QA pair:
ID: temporal_qa_227_with_llmeval.json_9_1
File: temporal_qa_227_with_llmeval.json | Event: 9 | QA: 1

Event Date: 2016-04-24
Event Summary: Following an appeal, 12-year-old Dima al-Wawi is freed from an Israeli prison. She is believed to be the youngest female Palestinian ever imprisoned. Israeli civil law, which does not allow anyone under 14 to go to jail, did not affect her imprisonment. Palestinian residents in the occupied territories are subject to military law that can sentence suspects as young as 12 to prison. (AP via ABC News)

Question Type: Entity Tracking

Question: Where is the 12-year-old girl living now after being released from prison?

Choices:
    A: East Jerusalem
  ✓ B: West Bank
    C: Ramallah
    D: Gaza Strip

Correct Answer: B

Referenced Article: Halhul, occupied West Bank - At the Jbara checkpoint near Tulkarem, the al-Wawi family and human rights groups' representatives waited patiently. The 12-year-old finally emerged, after havin

## Cell 4: Analysis Functions - Compare LLM vs Human Scores

**Run this cell after annotating to analyze agreement.**

This cell provides functions to:
- **`analyze_llm_human_agreement()`** - Calculate agreement statistics between LLM and human scores
- **`find_disagreements()`** - List QA pairs where scores don't match
- **`export_agreement_report()`** - Export detailed agreement report to JSON

**Use this cell to:**
1. Check overall agreement between your scores and LLM scores
2. Identify QA pairs where you disagree with the LLM
3. Generate reports for analysis

The cell will automatically run analysis if you have any annotations.

In [5]:
# Analysis functions for comparing LLM vs Human scores

def analyze_llm_human_agreement(qa_data: List[Dict[str, Any]]):
    """
    Analyze agreement between LLM scores and human scores.
    Returns detailed statistics.
    """
    annotated_pairs = [e for e in qa_data if e.get('human_annotated', False)]
    
    if not annotated_pairs:
        print("No annotated pairs found. Please annotate some QA pairs first.")
        return None
    
    total = len(annotated_pairs)
    
    # Full agreement (all three scores match)
    full_agreements = 0
    # Partial agreement (some scores match)
    partial_agreements = 0
    # Complete disagreement (no scores match)
    disagreements = 0
    
    # Score-by-score agreement
    question_agreements = 0
    answer_agreements = 0
    choice_agreements = 0
    
    # Detailed comparison
    comparison_details = []
    
    for e in annotated_pairs:
        llm_q = e.get('llm_question_score')
        llm_a = e.get('llm_answer_score')
        llm_c = e.get('llm_choice_score')
        human_q = e.get('human_question_score')
        human_a = e.get('human_answer_score')
        human_c = e.get('human_choice_score')
        
        q_match = (llm_q == human_q)
        a_match = (llm_a == human_a)
        c_match = (llm_c == human_c)
        
        if q_match:
            question_agreements += 1
        if a_match:
            answer_agreements += 1
        if c_match:
            choice_agreements += 1
        
        if q_match and a_match and c_match:
            full_agreements += 1
        elif q_match or a_match or c_match:
            partial_agreements += 1
        else:
            disagreements += 1
        
        comparison_details.append({
            'unique_id': e['unique_id'],
            'llm_scores': f"{llm_q}/{llm_a}/{llm_c}",
            'human_scores': f"{human_q}/{human_a}/{human_c}",
            'question_match': q_match,
            'answer_match': a_match,
            'choice_match': c_match,
            'full_match': q_match and a_match and c_match
        })
    
    results = {
        'total_annotated': total,
        'full_agreements': full_agreements,
        'partial_agreements': partial_agreements,
        'disagreements': disagreements,
        'question_agreements': question_agreements,
        'answer_agreements': answer_agreements,
        'choice_agreements': choice_agreements,
        'details': comparison_details
    }
    
    # Print summary
    print("="*80)
    print("LLM vs HUMAN SCORE AGREEMENT ANALYSIS")
    print("="*80)
    print(f"Total annotated pairs: {total}")
    print(f"\nFull Agreement (all 3 scores match):")
    print(f"  Count: {full_agreements}/{total} ({full_agreements/total*100:.1f}%)")
    print(f"\nPartial Agreement (1-2 scores match):")
    print(f"  Count: {partial_agreements}/{total} ({partial_agreements/total*100:.1f}%)")
    print(f"\nComplete Disagreement (no scores match):")
    print(f"  Count: {disagreements}/{total} ({disagreements/total*100:.1f}%)")
    print(f"\nScore-by-Score Agreement:")
    print(f"  Question score: {question_agreements}/{total} ({question_agreements/total*100:.1f}%)")
    print(f"  Answer score: {answer_agreements}/{total} ({answer_agreements/total*100:.1f}%)")
    print(f"  Choice score: {choice_agreements}/{total} ({choice_agreements/total*100:.1f}%)")
    print("="*80)
    
    return results

def find_disagreements(qa_data: List[Dict[str, Any]], 
                       show_details: bool = True,
                       max_display: int = 10):
    """
    Find and display QA pairs where human and LLM scores disagree.
    """
    disagreements = []
    
    for e in qa_data:
        if e.get('human_annotated', False):
            llm_q = e.get('llm_question_score')
            llm_a = e.get('llm_answer_score')
            llm_c = e.get('llm_choice_score')
            human_q = e.get('human_question_score')
            human_a = e.get('human_answer_score')
            human_c = e.get('human_choice_score')
            
            if not (llm_q == human_q and llm_a == human_a and llm_c == human_c):
                disagreements.append(e)
    
    if not disagreements:
        print("No disagreements found! All human scores match LLM scores.")
        return []
    
    print(f"Found {len(disagreements)} QA pairs with score disagreements:")
    print("="*80)
    
    for i, e in enumerate(disagreements[:max_display], 1):
        print(f"\n{i}. ID: {e['unique_id']}")
        print(f"   LLM scores:   Q={e.get('llm_question_score')}, A={e.get('llm_answer_score')}, C={e.get('llm_choice_score')}")
        print(f"   Human scores: Q={e.get('human_question_score')}, A={e.get('human_answer_score')}, C={e.get('human_choice_score')}")
        if e.get('human_notes'):
            print(f"   Notes: {e['human_notes']}")
        if show_details:
            print(f"   Question: {e.get('question', '')[:100]}...")
    
    if len(disagreements) > max_display:
        print(f"\n... and {len(disagreements) - max_display} more disagreements")
    
    return disagreements

def export_agreement_report(qa_data: List[Dict[str, Any]], 
                           output_path: str = None):
    """
    Export a detailed agreement report to JSON.
    """
    if output_path is None:
        output_path = os.path.join(OUTPUT_DIR, 'llm_human_agreement_report.json')
    
    results = analyze_llm_human_agreement(qa_data)
    
    if results:
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
        print(f"\nAgreement report saved to: {output_path}")
    
    return results

# Run analysis if there are any annotations
annotated_count = sum(1 for e in all_qa_data if e.get('human_annotated', False))
if annotated_count > 0:
    print(f"Found {annotated_count} annotated pairs. Running analysis...\n")
    analyze_llm_human_agreement(all_qa_data)
else:
    print("No annotations yet. Run this cell again after annotating some QA pairs.")

No annotations yet. Run this cell again after annotating some QA pairs.
